# Lab 4: Regression Analysis with Regularization Techniques

**Name:** Rajiv  
**Course:** Advanced Big Data and Data Mining (MSCS-634-B01)  
**Lab Assignment:** Lab 4 — Regression Analysis with Regularization Techniques  
**Date:** March 28, 2026

---

## Overview

In this lab, we explore multiple regression techniques applied to the **Diabetes Dataset** from `sklearn.datasets`. We implement:

1. **Simple Linear Regression** — using a single feature
2. **Multiple Regression** — using all features
3. **Polynomial Regression** — extending features with polynomial terms
4. **Ridge Regression** — L2 regularization
5. **Lasso Regression** — L1 regularization

Models are evaluated using **MAE**, **MSE**, **RMSE**, and **R²**, with visualizations throughout.

---
## Section 1: Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn datasets and preprocessing
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# Regression models
from sklearn.linear_model import LinearRegression, Ridge, Lasso

# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Plot styling
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13
sns.set_style("whitegrid")

print("All libraries imported successfully.")

---
## Section 2: Data Loading and Exploration

We load the **Diabetes Dataset**, which contains 10 numeric features (age, sex, BMI, blood pressure, and six blood serum measurements) measured on 442 patients, along with a quantitative measure of disease progression one year after baseline as the target.

In [ ]:
# Load dataset
diabetes = load_diabetes()

# Build a DataFrame for easy exploration
df = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
df['target'] = diabetes.target

print("Dataset Shape:", df.shape)
print("\nFeature Names:", diabetes.feature_names)
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Descriptive statistics
print("Descriptive Statistics:")
df.describe()

In [ ]:
# Distribution of target variable
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['target'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Target (Disease Progression)')
axes[0].set_xlabel('Target Value')
axes[0].set_ylabel('Frequency')

# Distribution of all features
df.drop('target', axis=1).hist(bins=20, figsize=(14, 8), color='steelblue', edgecolor='white', layout=(2, 5))
plt.suptitle('Feature Distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Correlation heatmap
plt.figure(figsize=(12, 8))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap of Features and Target')
plt.tight_layout()
plt.show()

print("\nTop features correlated with target:")
print(corr['target'].drop('target').sort_values(ascending=False))

---
## Section 3: Data Preparation and Cleaning

We check for missing values, define feature/target arrays, and split into training and testing sets (80/20 split). The Diabetes dataset features are already standardized by sklearn, so additional scaling is not required for linear models.

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

# Define features (X) and target (y)
X = diabetes.data          # shape (442, 10)
y = diabetes.target        # shape (442,)

# Train/test split — 80% train, 20% test, fixed random state for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\nTraining set size : {X_train.shape}")
print(f"Testing  set size : {X_test.shape}")

# Helper function: compute and print metrics
def evaluate_model(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_true, y_pred)
    print(f"\n{'='*40}")
    print(f"Model : {name}")
    print(f"  MAE  : {mae:.4f}")
    print(f"  MSE  : {mse:.4f}")
    print(f"  RMSE : {rmse:.4f}")
    print(f"  R²   : {r2:.4f}")
    return {'Model': name, 'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R²': r2}

# Storage for all model results
results = []

---
## Section 4: Simple Linear Regression

We use the single most correlated feature with the target — **bmi** — as the independent variable. This produces the simplest possible model and gives a clear 2D visualization of the regression line.

In [ ]:
# BMI is the 3rd feature (index 2) and has the highest positive correlation
bmi_idx = diabetes.feature_names.tolist().index('bmi')

X_train_slr = X_train[:, bmi_idx].reshape(-1, 1)
X_test_slr  = X_test[:, bmi_idx].reshape(-1, 1)

# Train Simple Linear Regression
slr = LinearRegression()
slr.fit(X_train_slr, y_train)
y_pred_slr = slr.predict(X_test_slr)

# Evaluate
res_slr = evaluate_model("Simple Linear Regression (BMI)", y_test, y_pred_slr)
results.append(res_slr)

print(f"\n  Coefficient (slope) : {slr.coef_[0]:.4f}")
print(f"  Intercept           : {slr.intercept_:.4f}")

# Visualization
plt.figure(figsize=(10, 5))
# Sort for a clean regression line
sort_idx = np.argsort(X_test_slr[:, 0])
plt.scatter(X_test_slr, y_test, color='steelblue', alpha=0.7, label='Actual')
plt.plot(X_test_slr[sort_idx], y_pred_slr[sort_idx], color='crimson', linewidth=2, label='Regression Line')
plt.xlabel('BMI (standardized)')
plt.ylabel('Disease Progression (target)')
plt.title('Simple Linear Regression: BMI vs Disease Progression')
plt.legend()
plt.tight_layout()
plt.show()

---
## Section 5: Multiple Regression

We now use **all 10 features** to train a Multiple Regression model. Using more predictors generally reduces bias and can improve R², but also risks overfitting on small datasets.

In [ ]:
# Train Multiple Linear Regression (all features)
mlr = LinearRegression()
mlr.fit(X_train, y_train)
y_pred_mlr = mlr.predict(X_test)

# Evaluate
res_mlr = evaluate_model("Multiple Linear Regression (All Features)", y_test, y_pred_mlr)
results.append(res_mlr)

print("\nFeature Coefficients:")
coeff_df = pd.DataFrame({'Feature': diabetes.feature_names, 'Coefficient': mlr.coef_})
print(coeff_df.sort_values('Coefficient', ascending=False).to_string(index=False))

# Visualization: Predicted vs Actual
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_mlr, color='darkorange', alpha=0.7, edgecolors='k', linewidth=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2, label='Perfect Prediction')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Multiple Regression: Predicted vs Actual')
plt.legend()
plt.tight_layout()
plt.show()

---
## Section 6: Polynomial Regression

We extend the BMI feature with polynomial terms to capture non-linear relationships. We test degrees **2, 3, and 5** and observe how higher degrees lead to overfitting (training R² increases but test R² may drop).

In [ ]:
degrees = [2, 3, 5]
poly_results = []

fig, axes = plt.subplots(1, len(degrees), figsize=(18, 5))
# For plotting a smooth curve along BMI range
X_plot = np.linspace(X_test_slr.min(), X_test_slr.max(), 300).reshape(-1, 1)

for i, deg in enumerate(degrees):
    poly = PolynomialFeatures(degree=deg, include_bias=False)

    # Transform single feature (BMI)
    X_train_poly = poly.fit_transform(X_train_slr)
    X_test_poly  = poly.transform(X_test_slr)
    X_plot_poly  = poly.transform(X_plot)

    lr_poly = LinearRegression()
    lr_poly.fit(X_train_poly, y_train)
    y_pred_poly = lr_poly.predict(X_test_poly)
    y_plot_poly = lr_poly.predict(X_plot_poly)

    # Metrics
    r = evaluate_model(f"Polynomial Regression (degree={deg})", y_test, y_pred_poly)
    poly_results.append(r)

    # Only add the best degree (degree=2) to main results for comparison
    if deg == 2:
        results.append(r)

    # Plot
    axes[i].scatter(X_test_slr, y_test, color='steelblue', alpha=0.6, label='Actual', s=25)
    axes[i].plot(X_plot, y_plot_poly, color='crimson', linewidth=2, label=f'Degree {deg}')
    axes[i].set_title(f'Polynomial Degree {deg}\nR²={r["R²"]:.3f}')
    axes[i].set_xlabel('BMI (standardized)')
    axes[i].set_ylabel('Target')
    axes[i].legend(fontsize=8)

plt.suptitle('Polynomial Regression — Varying Degrees', fontsize=14)
plt.tight_layout()
plt.show()

# Summary table for polynomial degrees
print("\nPolynomial Regression Summary across Degrees:")
poly_df = pd.DataFrame(poly_results)
print(poly_df[['Model', 'MAE', 'MSE', 'RMSE', 'R²']].to_string(index=False))

**Observation:** As polynomial degree increases, the curve becomes more flexible and hugs training data more tightly, potentially overfitting. Degree 2 typically achieves the best balance on the test set for this dataset.

---
## Section 7: Ridge and Lasso Regression

**Ridge (L2)** adds a penalty proportional to the **square** of coefficients, shrinking them toward zero but keeping all features.  
**Lasso (L1)** adds a penalty proportional to the **absolute value** of coefficients, which can drive some coefficients to exactly zero — performing implicit feature selection.

We test multiple alpha values to observe the regularization effect.

In [ ]:
alphas = [0.01, 0.1, 1.0, 10.0, 100.0]

ridge_r2 = []
lasso_r2 = []

print("Effect of Alpha on Ridge and Lasso R² (test set):\n")
print(f"{'Alpha':>8} | {'Ridge R²':>10} | {'Lasso R²':>10}")
print("-" * 35)

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train, y_train)
    r2_ridge = r2_score(y_test, ridge.predict(X_test))
    ridge_r2.append(r2_ridge)

    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_train, y_train)
    r2_lasso = r2_score(y_test, lasso.predict(X_test))
    lasso_r2.append(r2_lasso)

    print(f"{alpha:>8} | {r2_ridge:>10.4f} | {r2_lasso:>10.4f}")

# Alpha effect plot
plt.figure(figsize=(10, 5))
plt.plot(alphas, ridge_r2, marker='o', color='royalblue', label='Ridge R²', linewidth=2)
plt.plot(alphas, lasso_r2, marker='s', color='tomato',    label='Lasso R²', linewidth=2)
plt.xscale('log')
plt.xlabel('Alpha (log scale)')
plt.ylabel('R² Score')
plt.title('Ridge vs Lasso: R² Score vs Alpha')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Train final Ridge and Lasso models with alpha=1.0 (a common default)
ridge_final = Ridge(alpha=1.0)
ridge_final.fit(X_train, y_train)
y_pred_ridge = ridge_final.predict(X_test)

lasso_final = Lasso(alpha=0.1, max_iter=10000)
lasso_final.fit(X_train, y_train)
y_pred_lasso = lasso_final.predict(X_test)

# Evaluate and store
res_ridge = evaluate_model("Ridge Regression (alpha=1.0)", y_test, y_pred_ridge)
res_lasso = evaluate_model("Lasso Regression (alpha=0.1)", y_test, y_pred_lasso)
results.append(res_ridge)
results.append(res_lasso)

# Coefficient comparison
coeff_compare = pd.DataFrame({
    'Feature'  : diabetes.feature_names,
    'MLR'      : mlr.coef_,
    'Ridge(α=1)': ridge_final.coef_,
    'Lasso(α=0.1)': lasso_final.coef_
})
print("\nCoefficient Comparison (MLR vs Ridge vs Lasso):")
print(coeff_compare.to_string(index=False))

# Bar chart of coefficients
x = np.arange(len(diabetes.feature_names))
width = 0.28

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x - width, coeff_compare['MLR'],        width, label='MLR',         color='steelblue')
ax.bar(x,         coeff_compare['Ridge(α=1)'], width, label='Ridge(α=1)',  color='darkorange')
ax.bar(x + width, coeff_compare['Lasso(α=0.1)'], width, label='Lasso(α=0.1)', color='seagreen')
ax.set_xticks(x)
ax.set_xticklabels(diabetes.feature_names, rotation=30)
ax.set_ylabel('Coefficient Value')
ax.set_title('Feature Coefficients: MLR vs Ridge vs Lasso')
ax.legend()
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
# Predicted vs Actual: Ridge and Lasso side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, title, color in [
    (axes[0], y_pred_ridge, 'Ridge Regression (α=1.0)', 'royalblue'),
    (axes[1], y_pred_lasso, 'Lasso Regression (α=0.1)', 'tomato')
]:
    ax.scatter(y_test, y_pred, color=color, alpha=0.7, edgecolors='k', linewidth=0.4, s=40)
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax.plot(lims, lims, 'k--', linewidth=1.5, label='Perfect Prediction')
    ax.set_xlabel('Actual Values')
    ax.set_ylabel('Predicted Values')
    ax.set_title(f'{title}\nPredicted vs Actual')
    ax.legend()

plt.tight_layout()
plt.show()

---
## Section 8: Model Comparison and Analysis

We compile all model results into a summary table and visualize metric comparisons.

In [ ]:
# Build comparison DataFrame
comparison_df = pd.DataFrame(results)
comparison_df = comparison_df.set_index('Model')
comparison_df = comparison_df.round(4)

print("=" * 75)
print("MODEL PERFORMANCE COMPARISON SUMMARY")
print("=" * 75)
print(comparison_df.to_string())
print("=" * 75)

In [ ]:
# Bar chart comparison for all metrics
metrics = ['MAE', 'MSE', 'RMSE', 'R²']
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

colors = plt.cm.tab10.colors

for i, metric in enumerate(metrics):
    vals = comparison_df[metric]
    bars = axes[i].barh(vals.index, vals.values, color=colors[:len(vals)])
    axes[i].set_title(f'Model Comparison — {metric}')
    axes[i].set_xlabel(metric)
    for bar, val in zip(bars, vals.values):
        axes[i].text(bar.get_width() + (vals.max() * 0.01), bar.get_y() + bar.get_height() / 2,
                     f'{val:.4f}', va='center', fontsize=8)
    axes[i].invert_yaxis()

plt.suptitle('Regression Model Comparison Across All Metrics', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap of normalized metrics for visual comparison
from sklearn.preprocessing import MinMaxScaler

norm_df = comparison_df[['MAE', 'MSE', 'RMSE']].copy()
# Normalize error metrics (lower is better) — invert R² so all "better" = lower
norm_df['1 - R²'] = 1 - comparison_df['R²']

scaler = MinMaxScaler()
norm_values = pd.DataFrame(
    scaler.fit_transform(norm_df),
    index=norm_df.index,
    columns=norm_df.columns
)

plt.figure(figsize=(10, 5))
sns.heatmap(norm_values, annot=True, fmt='.3f', cmap='RdYlGn_r',
            linewidths=0.5, cbar_kws={'label': 'Normalized Score (lower = better)'})
plt.title('Normalized Performance Heatmap (lower is better for all metrics)')
plt.tight_layout()
plt.show()

---
## Key Observations and Conclusions

### 1. Simple Linear Regression (BMI only)
- Using just BMI as a predictor achieves a modest R² (~0.35–0.40). BMI is the most linearly correlated feature, but a single feature cannot fully explain disease progression.
- The regression line clearly shows a positive relationship but with considerable scatter.

### 2. Multiple Linear Regression (All Features)
- Adding all 10 features significantly improves performance (R² ≈ 0.45–0.53). This confirms that disease progression is a multi-factorial outcome.
- The predicted-vs-actual scatter plot shows better alignment with the identity line compared to the simple model.

### 3. Polynomial Regression
- Degree 2 typically matches or slightly improves upon Simple Linear Regression on the test set.
- Higher degrees (3, 5) show the beginnings of overfitting — the curve fits the training data more tightly but may not generalize as well, evidenced by worsening test metrics.
- This illustrates the classic **bias-variance tradeoff**: simpler models underfit, overly complex models overfit.

### 4. Ridge Regression
- Ridge performs comparably to Multiple Linear Regression for moderate alpha values (e.g., α=1.0). The L2 penalty shrinks coefficients smoothly without eliminating features.
- At very high alpha values (e.g., 100), excessive shrinkage hurts performance. At very low alpha values, it approaches ordinary least squares.

### 5. Lasso Regression
- Lasso's L1 penalty drives some feature coefficients to **exactly zero**, effectively performing automatic feature selection. This is visible in the coefficient comparison chart.
- For this dataset, an alpha of 0.1 provides a good balance, retaining the most predictive features while suppressing noise from less relevant ones.
- Lasso can be particularly useful when interpretability and feature selection are priorities.

### 6. Overall Insights
- **Multiple Linear Regression** offers the best all-around performance with no regularization needed for this moderately-sized, well-conditioned dataset.
- **Ridge and Lasso** provide slight improvements in generalization and are especially valuable when datasets are high-dimensional or features are correlated.
- The **Diabetes dataset** confirms that BMI and serum measurements (s5, s4) are the strongest predictors of disease progression.
- Regularization helps prevent overfitting but requires careful tuning of the alpha hyperparameter.